# Lesson 6: Efficient coding models for orientation-to-value estimation

This lesson introduces the **efficient coding models** from *"Beyond perception: Multi-stage efficient coding of perceptual and value representations"* (Bedi, de Hollander, Harl & Ruff).

## The task

Subjects see a Gabor patch at a specific orientation (7.5-172.5 degrees) and must estimate the monetary value (2-42 CHF) that orientation maps to. The orientation-to-value mapping is learned beforehand. Crucially, the **same subject** works with **two different mappings** (CDF and inverse-CDF), creating different value distributions from the same perceptual inputs.

## The models

Three representational architectures are compared:

1. **Efficient-perception model**: Efficient coding + Bayesian decoding in orientation space only. Value = G(theta_hat).
2. **Efficient-valuation model**: Veridical perception. Efficient coding + Bayesian decoding in value space only.
3. **Sequential efficient-coding model**: Both stages — perceptual uncertainty propagates into the value stage.

The key constraint: **noise parameters are shared across mapping conditions** within a subject. Different mappings produce different behavioral patterns from the same noise, which identifies the model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from bauer.utils.data import load_abstract_values_pilot
from bauer.efficient_coding import (
    EfficientPerceptionModel, EfficientValuationModel, 
    SequentialEfficientCodingModel,
    orientation_to_value_np, MAPPING_ORIENTATIONS_DEG, MAPPING_VALUES,
    long_term_orientation_prior_np
)

## 1. The orientation-to-value mappings

The experiment uses three mappings (the pilot uses CDF and inverse-CDF):
- **CDF ("Misaligned")**: S-shaped, compresses values at the extremes
- **Inverse CDF ("Aligned")**: Inverted S, compresses values in the middle  
- **Linear ("Uniform")**: Near-linear mapping

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 1. Value mappings
ax = axes[0]
ori = np.linspace(0, 180, 200)
for name, label, color in [('cdf', 'CDF (Misaligned)', 'C1'), 
                             ('inverse_cdf', 'Inv. CDF (Aligned)', 'C2'),
                             ('linear', 'Linear (Uniform)', 'C0')]:
    ax.plot(ori, orientation_to_value_np(ori, name), label=label, color=color)
ax.set_xlabel('Orientation (deg)')
ax.set_ylabel('Value (CHF)')
ax.set_title('Orientation-to-value mappings')
ax.legend()

# 2. Long-term orientation prior
ax = axes[1]
phi = np.linspace(0, 2*np.pi, 200, endpoint=False)
prior = long_term_orientation_prior_np(phi)
ax.plot(phi * 180 / (2*np.pi), prior)  # convert back to degrees
ax.set_xlabel('Orientation (deg)')
ax.set_ylabel('Prior density')
ax.set_title('Long-term perceptual prior')
ax.axvline(90, ls='--', color='gray', alpha=0.5)

# 3. Induced value priors
ax = axes[2]
for name, label, color in [('cdf', 'CDF', 'C1'), ('inverse_cdf', 'Inv. CDF', 'C2'),
                             ('linear', 'Linear', 'C0')]:
    vals = orientation_to_value_np(np.linspace(7.5, 172.5, 1000), name)
    ax.hist(vals, bins=30, density=True, alpha=0.5, label=label, color=color)
ax.set_xlabel('Value (CHF)')
ax.set_ylabel('Density')
ax.set_title('Short-term value priors')
ax.legend()

plt.tight_layout()
plt.show()

## 2. Load pilot data

Each subject works with **both** CDF and inverse-CDF mappings. We use both conditions jointly — the noise parameters are shared, only the value mapping changes.

In [ ]:
df = load_abstract_values_pilot()
print(f'Loaded {len(df)} trials from {df.index.get_level_values("subject").nunique()} subjects')
print(f'Mappings: {df.index.get_level_values("mapping").unique().tolist()}')
print(f'Orientations: {sorted(df["orientation"].unique())}')

# Show one subject's data
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, mapping in zip(axes, ['cdf', 'inverse_cdf']):
    sub_data = df.xs(5, level='subject').xs(mapping, level='mapping')
    ax.scatter(sub_data['orientation'], sub_data['response'], alpha=0.3, s=10)
    ori = np.linspace(7.5, 172.5, 100)
    ax.plot(ori, orientation_to_value_np(ori, mapping), 'r-', lw=2, label='True mapping')
    ax.set_xlabel('Orientation (deg)')
    ax.set_ylabel('Response (CHF)')
    ax.set_title(f'Subject 5 — {mapping}')
    ax.legend()
plt.tight_layout()
plt.show()

## 3. Fitting the Efficient-Perception model

The simplest model: only perceptual noise (kappa_r). Value is deterministically computed as G(theta_hat).

We fit one subject using **both mapping conditions jointly** — the noise parameter is shared.

In [ ]:
# Prepare paradigm for subject 5 with BOTH mappings
sub5 = df.xs(5, level='subject').reset_index()
paradigm = sub5[['orientation', 'response', 'mapping']].copy()
paradigm.index = pd.MultiIndex.from_arrays(
    [np.ones(len(paradigm), dtype=int), range(len(paradigm))],
    names=['subject', 'trial'])

print(f'Total trials: {len(paradigm)}')
print(f'  CDF: {(paradigm["mapping"] == "cdf").sum()}')
print(f'  Inv. CDF: {(paradigm["mapping"] == "inverse_cdf").sum()}')

# Fit with different grid resolutions to check convergence
print('\n--- Grid resolution comparison ---')
for res in [31, 51, 101]:
    model = EfficientPerceptionModel(paradigm, grid_resolution=res,
                                     perceptual_prior='long_term')
    model.build_estimation_model(paradigm, hierarchical=False, flat_prior=True)
    pars = model.fit_map(progressbar=False)
    print(f'  grid_res={res}: kappa_r = {float(pars["kappa_r"]):.2f}')

### Posterior predictive checks — single subject

Let's check whether the fitted model can reproduce the observed response patterns under both mapping conditions.

In [ ]:
# Use the grid_resolution=51 fit
model = EfficientPerceptionModel(paradigm, grid_resolution=51,
                                 perceptual_prior='long_term')
model.build_estimation_model(paradigm, hierarchical=False, flat_prior=True)
pars = model.fit_map(progressbar=False)

# Predict and simulate
predictions = model.predict(paradigm, pars)
simulations = model.simulate(paradigm, pars, n_samples=5)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for col, mapping in enumerate(['cdf', 'inverse_cdf']):
    mask = predictions['mapping'] == mapping
    pred_m = predictions[mask]
    data_m = paradigm[paradigm['mapping'] == mapping]

    # --- Top: mean response ---
    ax = axes[0, col]
    emp_mean = data_m.groupby('orientation')['response'].mean()
    emp_se = data_m.groupby('orientation')['response'].sem()
    ax.errorbar(emp_mean.index, emp_mean.values, yerr=emp_se.values * 1.96,
                fmt='o', color='C0', label='Data (mean ± 95% CI)', zorder=3)

    pred_mean = pred_m.groupby('orientation')['predicted_mean'].mean()
    ax.plot(pred_mean.index, pred_mean.values, 's-', color='C1',
            label='Model prediction', markersize=4)

    ori = np.linspace(7.5, 172.5, 100)
    ax.plot(ori, orientation_to_value_np(ori, mapping), 'k--', alpha=0.3,
            label='True mapping')
    ax.set_xlabel('Orientation (deg)')
    ax.set_ylabel('Mean response (CHF)')
    ax.set_title(f'{mapping}')
    ax.legend(fontsize=8)

    # --- Bottom: bias (response - true value) ---
    ax = axes[1, col]
    # Empirical bias per orientation
    data_with_true = data_m.copy()
    data_with_true['true_value'] = orientation_to_value_np(
        data_with_true['orientation'].values, mapping)
    emp_bias = data_with_true.groupby('orientation').apply(
        lambda g: (g['response'] - g['true_value']).mean())

    ax.plot(emp_bias.index, emp_bias.values, 'o', color='C0', label='Data bias')

    # Model predicted bias
    pred_with_true = pred_m.copy()
    pred_with_true['true_value'] = orientation_to_value_np(
        pred_with_true['orientation'].values, mapping)
    model_bias = pred_with_true.groupby('orientation').apply(
        lambda g: (g['predicted_mean'] - g['true_value']).mean())
    ax.plot(model_bias.index, model_bias.values, 's-', color='C1',
            label='Model bias', markersize=4)

    ax.axhline(0, color='k', ls='--', alpha=0.3)
    ax.set_xlabel('Orientation (deg)')
    ax.set_ylabel('Valuation bias (CHF)')
    ax.set_title(f'{mapping} — bias')
    ax.legend(fontsize=8)

plt.suptitle(f'Subject 5 — Efficient-Perception model (κ_r = {float(pars["kappa_r"]):.1f})',
             fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Simulated data vs real data
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, mapping in zip(axes, ['cdf', 'inverse_cdf']):
    data_m = paradigm[paradigm['mapping'] == mapping]
    sim_m = simulations[simulations['mapping'] == mapping]

    # One simulated sample as background
    sim1 = sim_m.xs(1, level='sample')
    ax.scatter(sim1['orientation'], sim1['simulated_response'],
               alpha=0.15, s=8, color='C1', label='Simulated', zorder=1)
    ax.scatter(data_m['orientation'], data_m['response'],
               alpha=0.4, s=12, color='C0', label='Data', zorder=2)

    ori = np.linspace(7.5, 172.5, 100)
    ax.plot(ori, orientation_to_value_np(ori, mapping), 'k--', alpha=0.3)
    ax.set_xlabel('Orientation (deg)')
    ax.set_ylabel('Response (CHF)')
    ax.set_title(f'{mapping}')
    ax.legend(fontsize=9)

plt.suptitle('Real data vs simulated data from fitted model', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Fit all subjects with the Efficient-Perception model

In [ ]:
# Prepare multi-subject paradigm with both mappings
all_paradigms = []
for sub_id in df.index.get_level_values('subject').unique():
    sub_data = df.xs(sub_id, level='subject').reset_index()
    p = sub_data[['orientation', 'response', 'mapping']].copy()
    p['subject'] = sub_id
    all_paradigms.append(p)

paradigm_all = pd.concat(all_paradigms, ignore_index=True)
paradigm_all = paradigm_all.set_index(['subject', paradigm_all.groupby('subject').cumcount()])
paradigm_all.index.names = ['subject', 'trial']
print(f'All subjects: {len(paradigm_all)} trials')

# Fit individual MLE per subject (both conditions simultaneously)
model = EfficientPerceptionModel(paradigm_all, grid_resolution=51,
                                 perceptual_prior='long_term')
mle_results = model.fit_map_individual(paradigm_all, flat_prior=True)
print('\nFitted kappa_r per subject:')
print(mle_results.round(2))

## 5. Fitting the Sequential model (both perceptual and value noise)

The sequential model has two free parameters: `kappa_r` (perceptual precision) and `sigma_rep` (value noise). The perceptual uncertainty propagates into the value stage.

In [ ]:
# Fit sequential model to subject 5 (both mappings)
sub5 = df.xs(5, level='subject').reset_index()
paradigm = sub5[['orientation', 'response', 'mapping']].copy()
paradigm.index = pd.MultiIndex.from_arrays(
    [np.ones(len(paradigm), dtype=int), range(len(paradigm))],
    names=['subject', 'trial'])

seq_model = SequentialEfficientCodingModel(paradigm, grid_resolution=31,
                                            perceptual_prior='long_term')
seq_model.build_estimation_model(paradigm, hierarchical=False, flat_prior=True)
pars = seq_model.fit_map(progressbar=False)
print(f'Sequential model fit:')
print(f'  kappa_r = {float(pars["kappa_r"]):.2f}')
print(f'  sigma_rep = {float(pars["sigma_rep"]):.4f}')

### Sequential model: posterior predictive checks

Compare the perception-only and sequential models side by side.

In [ ]:
# Get predictions from both models
perc_pred = model.predict(paradigm, model.fit_map(progressbar=False))
seq_pred = seq_model.predict(paradigm, pars)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for col, mapping in enumerate(['cdf', 'inverse_cdf']):
    data_m = paradigm[paradigm['mapping'] == mapping]
    data_m = data_m.copy()
    data_m['true_value'] = orientation_to_value_np(
        data_m['orientation'].values, mapping)

    # --- Top: mean response ---
    ax = axes[0, col]
    emp_mean = data_m.groupby('orientation')['response'].mean()
    emp_se = data_m.groupby('orientation')['response'].sem()
    ax.errorbar(emp_mean.index, emp_mean.values, yerr=emp_se.values * 1.96,
                fmt='o', color='C0', label='Data', zorder=3, markersize=5)

    for pred, name, color in [(perc_pred, 'Perception only', 'C1'),
                               (seq_pred, 'Sequential', 'C2')]:
        pm = pred[pred['mapping'] == mapping].groupby('orientation')['predicted_mean'].mean()
        ax.plot(pm.index, pm.values, '-', color=color, label=name)

    ori = np.linspace(7.5, 172.5, 100)
    ax.plot(ori, orientation_to_value_np(ori, mapping), 'k--', alpha=0.3)
    ax.set_xlabel('Orientation (deg)')
    ax.set_ylabel('Mean response (CHF)')
    ax.set_title(f'{mapping}')
    ax.legend(fontsize=8)

    # --- Bottom: bias ---
    ax = axes[1, col]
    emp_bias = data_m.groupby('orientation').apply(
        lambda g: (g['response'] - g['true_value']).mean())
    ax.plot(emp_bias.index, emp_bias.values, 'o', color='C0', label='Data', markersize=5)

    for pred, name, color in [(perc_pred, 'Perception only', 'C1'),
                               (seq_pred, 'Sequential', 'C2')]:
        p = pred[pred['mapping'] == mapping].copy()
        p['true_value'] = orientation_to_value_np(p['orientation'].values, mapping)
        mb = p.groupby('orientation').apply(
            lambda g: (g['predicted_mean'] - g['true_value']).mean())
        ax.plot(mb.index, mb.values, '-', color=color, label=name)

    ax.axhline(0, color='k', ls='--', alpha=0.3)
    ax.set_xlabel('Orientation (deg)')
    ax.set_ylabel('Valuation bias (CHF)')
    ax.set_title(f'{mapping} — bias')
    ax.legend(fontsize=8)

plt.suptitle('Perception-only vs Sequential model — Subject 5', fontsize=13)
plt.tight_layout()
plt.show()